In [ ]:
# Make sure reticulate is installed.
# Separately, you might have to configure reticulate to use the correct version of Python
# (say, if you have `bc2` installed in a virtual environment).
install.packages("reticulate")

In [ ]:
# Load the interface to the `bc2.evaluate` Python library.
# This package provides R bindings for the Azure model training infrastructure.
source("bc2-client.R")

# Connect to the Azure services where the documents / models live.
# We need both the Blob Storage service and also the Form Recognizer service.
# You will need to pass `blob_api_key` and `form_api_key` if you do not have
# Azure Command Line tools / `az login` configured.
connect_to_az(blob_account_url="https://blindchargingdev.blob.core.windows.net/",
              blob_container="bcdev",
              form_endpoint="https://bc-formr-dev.cognitiveservices.azure.com/")

In [ ]:
## Example use of all the available functions.

# Query to find which docs are available for training in the given directory.
docs <- list_docs("autoeval")

# Queries to find available models.
extractors <- list_extraction_models()
classifiers <- list_classifiers()

# Train an extraction model. (Takes a long time! only uncomment if you want to wait ~30 mins)
# train_extraction_model("test-from-R", filter(docs, has_labels)$name)

# Train a classifier. These go pretty quickly! Just make sure the name is unique.
train_classifier("test-classify-from-R", docs$name, docs$has_labels)

# Run the model on a couple of documents.
extraction <- run_extraction_model("test-from-R",
                                   c("autoeval/17-182586_Assault_Rpt_R_page_4.pdf",
                                     "autoeval/16-187153_Assault_Rpt_R_page_4.pdf"))

# Load ground-truth labeled documents (for extraction model).
labeled <- load_true_labels(c("autoeval/17-182586_Assault_Rpt_R_page_4.pdf",
                              "autoeval/16-187153_Assault_Rpt_R_page_4.pdf"))

# Visualize label bounding boxes overlaid on an image of the document.
show_bounds("autoeval/17-182586_Assault_Rpt_R_page_4.pdf", labeled[1,], extraction[1,])

# Run the classifier on a couple of documents.
classification <- run_classifier("test-classify-from-R",
                                 c("autoeval/17-182586_Assault_Rpt_R_page_4.pdf",
                                   "autoeval/16-187153_Assault_Rpt_R_page_4.pdf"))